In [2]:
import pandas as pd
import torch
import gc
import re
import os
from tqdm import tqdm
from ultralytics import YOLO
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from PIL import Image

# ==========================================
# 1. 테스트 데이터 로드 및 정답지 준비
# ==========================================
test_df = pd.read_csv('test_class.csv')

# 최종 제출용 정답을 담을 딕셔너리 { 'test_0001.jpg': 'a', ... }
final_answers = {}

# ==========================================
# 2. [Track A] YOLOv10 추론 (개수 세기 특화)
# ==========================================
print("🚀 [Phase 1] YOLOv10-S 모델 추론을 시작합니다...")
yolo_df = test_df[test_df['class'] == 'YOLO']

# 학습된 YOLO 최고 성능 가중치 로드
yolo_model = YOLO('runs/detect/VQA_YOLO/yolov10_run_1/weights/best.pt')

for idx, row in tqdm(yolo_df.iterrows(), total=len(yolo_df)):
    img_path = str(row['path'])
    
    # 모델 추론 (박스 찾기)
    results = yolo_model(img_path, verbose=False)
    # 발견된 객체(박스)의 개수를 셈
    detected_count = len(results[0].boxes)
    
    # 💡 [핵심 로직] 탐지된 개수와 가장 가까운 숫자를 가진 선택지 찾기
    options = {'a': str(row['a']), 'b': str(row['b']), 'c': str(row['c']), 'd': str(row['d'])}
    best_choice = 'a'
    min_diff = float('inf')
    
    for key, text in options.items():
        # 선택지 텍스트에서 숫자만 추출 (예: "3개" -> 3)
        nums = re.findall(r'\d+', text)
        if nums:
            opt_num = int(nums[0])
            diff = abs(opt_num - detected_count)
            # 예측한 개수와 가장 오차가 적은 선택지를 정답으로 채택
            if diff < min_diff:
                min_diff = diff
                best_choice = key
                
    final_answers[row['id']] = best_choice

# 🧹 YOLO 퇴장 및 메모리 완벽 청소
del yolo_model
gc.collect()
torch.cuda.empty_cache()
print("✅ YOLO 추론 완료 및 VRAM 반환 성공!\n")

# ==========================================
# 3. [Track B] Qwen2-VL 추론 (종합 추론 특화)
# ==========================================
print("🚀 [Phase 2] Qwen2-VL 모델 추론을 시작합니다...")
# VLM과 빈칸(NaN)으로 남겨진 모든 데이터를 VLM이 풉니다.
vlm_df = test_df[test_df['class'].fillna('VLM') != 'YOLO']

# 파인튜닝된 최종 모델과 프로세서 로드 (OOM 방지를 위해 bfloat16 사용)
model_dir = "VQA_Qwen_LoRA/final_model"
vlm_model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_dir, 
    device_map="auto", 
    torch_dtype=torch.bfloat16
)
processor = AutoProcessor.from_pretrained(model_dir)

for idx, row in tqdm(vlm_df.iterrows(), total=len(vlm_df)):
    img_path = str(row['path'])
    q_text = str(row['question'])
    
    # 학습할 때 사용했던 프롬프트 구조와 100% 동일하게 구성
    prompt = (
        f"질문: {q_text}\n"
        f"a. {row['a']}\n"
        f"b. {row['b']}\n"
        f"c. {row['c']}\n"
        f"d. {row['d']}\n"
        f"정답:"
    )
    
    messages = [
        {"role": "user", "content": [
            {"type": "image", "image": img_path},
            {"type": "text", "text": prompt}
        ]}
    ]
    
    # 프롬프트와 이미지를 모델이 이해할 수 있게 전처리
    text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image = Image.open(img_path).convert("RGB")
    
    inputs = processor(
        text=[text_prompt], 
        images=[image], 
        padding=True, 
        return_tensors="pt"
    ).to("cuda")
    
    # 모델 답변 생성 (최대 2토큰만 생성하도록 제한하여 알파벳만 빠르게 뽑아냄)
    with torch.no_grad():
        output_ids = vlm_model.generate(**inputs, max_new_tokens=2)
        
    # 입력 프롬프트 부분을 제외하고 새로 생성된 텍스트(알파벳)만 추출
    generated_ids = output_ids[0][inputs.input_ids.shape[1]:]
    answer = processor.decode(generated_ids, skip_special_tokens=True).strip().lower()
    
    # 만약 생성된 답이 a, b, c, d가 아니라면 안전장치로 a를 할당 (안전망)
    if answer not in ['a', 'b', 'c', 'd']:
        # 답변에 a,b,c,d가 포함되어있는지 한번 더 검사
        found = [ch for ch in ['a', 'b', 'c', 'd'] if ch in answer]
        answer = found[0] if found else 'a'
        
    final_answers[row['id']] = answer

print("✅ Qwen-VL 추론 완료!\n")

# ==========================================
# 4. 최종 앙상블 및 제출용 CSV 생성
# ==========================================
# 원본 test_df의 순서에 맞춰서 정답 매핑
test_df['answer'] = test_df['id'].map(final_answers)

# id와 answer 컬럼만 남겨서 제출 규격에 맞게 저장
submission_df = test_df[['id', 'answer']]
submission_df.to_csv('submission.csv', index=False)

print("🎉 모든 추론이 끝났습니다! 'submission.csv' 파일이 성공적으로 생성되었습니다.")
print("대회 플랫폼에 제출하여 우승을 확인하세요! 🏆")

🚀 [Phase 1] YOLOv10-S 모델 추론을 시작합니다...


100% 1709/1709 [00:53<00:00, 31.95it/s]
`torch_dtype` is deprecated! Use `dtype` instead!


✅ YOLO 추론 완료 및 VRAM 반환 성공!

🚀 [Phase 2] Qwen2-VL 모델 추론을 시작합니다...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

100% 3365/3365 [37:14<00:00,  1.51it/s]

✅ Qwen-VL 추론 완료!

🎉 모든 추론이 끝났습니다! 'submission.csv' 파일이 성공적으로 생성되었습니다.
대회 플랫폼에 제출하여 우승을 확인하세요! 🏆
